In [46]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("NousResearch/Hermes-3-Dataset")

In [47]:
# Or
from datasets import load_dataset

ds = load_dataset('cornell-movie-review-data/rotten_tomatoes')
type(ds)

datasets.dataset_dict.DatasetDict

In [48]:
ds['train'][0]

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
 'label': 1}

In [49]:
ds['train'][-1]

{'text': 'things really get weird , though not particularly scary : the movie is all portent and no content .',
 'label': 0}

In [50]:
# Load a specific split of the dataset
ds = load_dataset('cornell-movie-review-data/rotten_tomatoes', split='train')
ds[:2], len(ds)

({'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
   'the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson\'s expanded vision of j . r . r . tolkien\'s middle-earth .'],
  'label': [1, 1]},
 8530)

In [51]:
# Look the splits available in the dataset
from datasets import get_dataset_split_names
get_dataset_split_names('cornell-movie-review-data/rotten_tomatoes')

['train', 'validation', 'test']

## Iteratable Dataset

In [52]:
iteratable_dataset = load_dataset('cornell-movie-review-data/rotten_tomatoes', split='train', streaming=True)
for example in iteratable_dataset:
    print(example)
    break  # Just to show the first example

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


> You can also create an iteratable dataset from a local file or a directory of files.

```python
iterable_dataset = dataset.to_iterable_dataset()
```

In [53]:
next(iter(iteratable_dataset))

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
 'label': 1}

> What about loading a csv from a local file using datasets?

In [54]:
iris = load_dataset('csv', data_files='iris.csv')
iris

DatasetDict({
    train: Dataset({
        features: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species'],
        num_rows: 150
    })
})

In [55]:
iris['train'][-1]  # Access the last example in the train split

{'sepal_length': 5.9,
 'sepal_width': 3.0,
 'petal_length': 5.1,
 'petal_width': 1.8,
 'species': 'virginica'}

By default it takes the whole dataset as a single split. You can specify the split using the `split` parameter.

In [56]:
iris.shuffle(seeds=40)

DatasetDict({
    train: Dataset({
        features: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species'],
        num_rows: 150
    })
})

## Preprocessing the dataset

In [57]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
dataset = load_dataset('cornell-movie-review-data/rotten_tomatoes', split='train')

In [58]:
tokenizer(dataset[0]['text'])

{'input_ids': [101, 1996, 2600, 2003, 16036, 2000, 2022, 1996, 7398, 2301, 1005, 1055, 2047, 1000, 16608, 1000, 1998, 2008, 2002, 1005, 1055, 2183, 2000, 2191, 1037, 17624, 2130, 3618, 2084, 7779, 29058, 8625, 13327, 1010, 3744, 1011, 18856, 19513, 3158, 5477, 4168, 2030, 7112, 16562, 2140, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [59]:
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True)
dataset = dataset.map(tokenize_function, batched=True)

Making dataset format compatible with pytorch

In [60]:
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'label'])
dataset.format['type']

'torch'